# Q-Trust Phase 8 — Bank Pilot (First National Bank PQC Migration)

This notebook drives the end-to-end pilot implemented in `pilot/run_pilot.py`.
It simulates a mid-size bank migrating from classical to post-quantum cryptography:

1. Scan infrastructure → CBOM
2. Register CBOM on-chain
3. Quantum threat analysis (Shor's algorithm resource estimates)
4. GNN migration planner
5. Vendor attestation + migration + audit on-chain
6. Verify all on-chain records


In [ ]:
import sys
from pathlib import Path
_here = Path('.').resolve()
# The kernel's CWD is the notebook's own directory, so walk up to the
# repo root to locate pilot/run_pilot.py.
_root = next(
    (p.parent for p in (_here / 'run_pilot.py', _here / 'pilot' / 'run_pilot.py',
                      _here.parent / 'run_pilot.py', _here.parent / 'pilot' / 'run_pilot.py',
                      _here.parent.parent / 'run_pilot.py', _here.parent.parent / 'pilot' / 'run_pilot.py')
     if p.exists()),
    None,
)
if _root is None:
    raise RuntimeError(f'run_pilot.py not found near {_here}')
sys.path.insert(0, str(_root))
from run_pilot import step1_scan, step2_register, step3_quantum, step4_plan, step5_onchain, step6_verify  # noqa: E402 - after sys.path bootstrap
print(f'Pilot functions imported OK (repo root: {_root})')


## Step 1 — Scan infrastructure and build the CBOM

Scans a target host (TLS port 443, SSH port 22) and converts findings into a Cryptographic Bill of Materials.

In [ ]:
scan = step1_scan()
cbom = scan['cbom']
print('Assets in CBOM:', len(cbom.assets))

## Step 2 — Register the CBOM on-chain

Only the CBOM hash is stored on-chain; the full document is kept off-chain (IPFS / hash-only mode).

In [ ]:
asset_id = step2_register(cbom)
print('Registered asset id:', asset_id)

## Step 3 — Quantum threat analysis

Shor's algorithm resource estimates: physical qubits required to break RSA keys of each size, plotted against the IBM/Google quantum roadmap. Full simulation in `01_quantum_threat_demo.ipynb`.

In [ ]:
step3_quantum()

## Step 4 — GNN migration planner

The trained FIGNN/GCN model ranks the CBOM assets by migration priority and risk score.

In [ ]:
plan = step4_plan(cbom)
print('Top priority asset:', plan['migration_order'][0]['algorithm'])

## Step 5 — Attestation, migration, and audit on-chain

Vendor attests PQC readiness (ML-DSA-441) for the migrated product; the bank records the migration (classical → PQC) and an auditor posts a passing audit.

In [ ]:
step5_onchain(asset_id, plan, cbom)

## Step 6 — Verify every on-chain record

Independent verification: asset exists/active, product support check, org asset + migration lists.

In [ ]:
step6_verify(asset_id)
print('PILOT COMPLETE')